# Maching Learning

#### This notebook will clean the data, drop any irrelevant columns, and train a model to predict the time is takes to complete each ticket.

##### Note: please run this in Google Collab. When ran on compter CPU/GPU, values change even if seeds are set for the train/test split and regression model. This may be due to the difference in hardware and can sometimes cause a difference in results.

In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

In [ ]:
pd.set_option('display.max_columns', None)
df = pd.read_csv("jira_m2_accelerated.csv")

In [ ]:
df

,issue_key,issue_type,priority_numeric,status,component,story_points,sprint,assignee_id,reporter_id,resolution,num_comments,num_attachments,watchers_count,time_to_resolution_hours,num_transitions,description_length,ticket_age_days,has_labels,parent_epic_id,parent_story_id,has_child_stories,has_child_subtasks,priority,created_date,updated_date,problem_statement,description,cause
0,USR-11480,Sub-task,5,Code Review,Payment Processing,3,Sprint 17,41,15,Unresolved,7,1,1,41.09,3,28,39,0,NaN,USR-10463,0,0,Highest,2025-09-19,2025-10-17,File Upload experiencing form validation failures,QA testing identified data completeness affect...,Application logic in file upload missing data ...
1,APP-11312,Story,5,Closed,API Services,5,Sprint 14,7,83,Done,5,3,1,71.18,3,37,40,1,WEB-10522,NaN,0,0,Highest,2025-09-18,2025-10-07,Password Reset showing timeout errors,Users reporting timeout errors in password res...,Frontend optimization needed for password reset
2,MOB-10305,Bug,3,In Progress,API Services,3,Sprint 6,48,40,Unresolved,5,1,4,45.39,4,27,63,1,NaN,NaN,0,0,Medium,2025-08-26,2025-09-11,Data Export showing performance slowdown,Users reporting performance slowdown in data e...,Frontend optimization needed for data export
3,WEB-11545,Epic,4,In Progress,Analytics,1,Sprint 13,91,176,Unresolved,4,1,2,25.08,2,51,9,1,NaN,NaN,1,0,High,2025-10-19,2025-10-28,Review Submission showing browser compatibility,Users reporting browser compatibility in revie...,Frontend optimization needed for review submis...
4,API-11716,Bug,5,To Do,Payment Processing,5,Sprint 8,102,61,Unresolved,3,0,1,88.71,1,44,69,0,NaN,NaN,0,0,Highest,2025-08-20,2025-09-13,Subscription Management experiencing text trun...,QA testing identified incorrect data affecting...,Application logic in subscription management m...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,USR-10277,Task,4,Code Review,API Services,3,Sprint 2,22,50,Unresolved,6,1,1,22.47,2,29,13,1,NaN,NaN,0,0,High,2025-10-15,2025-11-02,Data Export showing browser compatibility,Users reporting browser compatibility in data ...,Frontend optimization needed for data export
1996,WEB-11841,Task,3,In Progress,User Management,2,Sprint 4,25,39,Unresolved,6,3,2,104.21,1,15,62,0,NaN,NaN,0,0,Medium,2025-08-27,2025-09-20,Password Reset showing loading delays,Users reporting loading delays in password res...,Frontend optimization needed for password reset
1997,APP-10255,Task,1,Closed,Customer Portal,2,Sprint 17,79,80,Done,8,1,1,76.46,3,15,98,1,NaN,NaN,0,0,Lowest,2025-07-22,2025-07-23,Account Settings showing browser compatibility,Users reporting browser compatibility in accou...,Frontend optimization needed for account settings
1998,APP-11344,Task,2,Code Review,Payment Processing,2,Sprint 20,73,194,Unresolved,5,2,0,94.66,4,61,15,0,NaN,NaN,0,0,Low,2025-10-13,2025-10-22,Customer Profile experiencing text truncation,QA testing identified data inconsistency affec...,Application logic in customer profile missing ...


## Data Cleaning

In [ ]:
# dropping unique columns as they would not be useful for modeling
df.drop(columns=["issue_key", "problem_statement", "description", "cause", "created_date", "updated_date"], inplace=True)

In [ ]:
# drop duplicate rows
df = df.drop_duplicates()

In [ ]:
df['sprint'] = pd.to_numeric(df['sprint'].str.extract('(\d+)', expand=False), errors='coerce').fillna(0).astype(int)

In [ ]:
# Define logical mapping for Ordinal Encoding
priority_order = {
    'Lowest': 1, 'Low': 2, 'Medium': 3, 'High': 4, 'Highest': 5
}

status_order = {
    'To Do': 1, 'Blocked': 2, 'In Progress': 3,
    'Code Review': 4, 'Testing': 5, 'Done': 6, 'Closed': 7
}

issue_type_order = {
    'Sub-task': 1, 'Task': 2, 'Bug': 3, 'Story': 4, 'Epic': 5
}

# 3. Apply the mappings
df['priority'] = df['priority'].map(priority_order)
df['status'] = df['status'].map(status_order)
df['issue_type'] = df['issue_type'].map(issue_type_order)


df['parent_epic_id'] = np.where(df['parent_epic_id'].notna(), 1, 0)
df['parent_story_id'] = np.where(df['parent_story_id'].notna(), 1, 0)

# 2. Update your categorical list to exclude these since they are now numeric
# Also excluding 'sprint', 'priority', 'status', and 'issue_type' as we handled them earlier
for col in ['component', 'resolution']:
    df[col] = df[col].astype('category').cat.codes

# Check the results
df.head()

,issue_type,priority_numeric,status,component,story_points,sprint,assignee_id,reporter_id,resolution,num_comments,num_attachments,watchers_count,time_to_resolution_hours,num_transitions,description_length,ticket_age_days,has_labels,parent_epic_id,parent_story_id,has_child_stories,has_child_subtasks,priority
0,1,5,4,7,3,17,41,15,1,7,1,1,41.09,3,28,39,0,0,1,0,0,5
1,4,5,7,0,5,14,7,83,0,5,3,1,71.18,3,37,40,1,1,0,0,0,5
2,3,3,3,0,3,6,48,40,1,5,1,4,45.39,4,27,63,1,0,0,0,0,3
3,5,4,3,1,1,13,91,176,1,4,1,2,25.08,2,51,9,1,0,0,1,0,4
4,3,5,1,7,5,8,102,61,1,3,0,1,88.71,1,44,69,0,0,0,0,0,5


## Target Variable

In [ ]:
y = df['time_to_resolution_hours']

X = df.drop(columns=['time_to_resolution_hours'])

## Baseline

In [ ]:
X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# XGBoost Model (Gradient Boosting)
xgboost_model_baseline = XGBRegressor(random_state=42)

# Train XGBoost
print("\n\nTraining XGBoost Model...")
xgboost_model_baseline.fit(
    X_train_baseline, y_train_baseline,
    eval_set=[(X_test_baseline, y_test_baseline)],
    verbose=30
)

xgboost_pred_baseline = xgboost_model_baseline.predict(X_test_baseline)

print("\n📊 XGBoost Regressor:")
print(f"  MSE:  {mean_squared_error(y_test_baseline, xgboost_pred_baseline):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test_baseline, xgboost_pred_baseline)):.4f}")
print(f"  MAE:  {mean_absolute_error(y_test_baseline, xgboost_pred_baseline):.4f}")
print(f"  R²:   {r2_score(y_test_baseline, xgboost_pred_baseline):.4f}")



Training XGBoost Model...
[0]	validation_0-rmse:19.36295
[30]	validation_0-rmse:21.11164
[60]	validation_0-rmse:21.60535
[90]	validation_0-rmse:21.77975
[99]	validation_0-rmse:21.83018

📊 XGBoost Regressor:
  MSE:  476.5568
  RMSE: 21.8302
  MAE:  17.7293
  R²:   -0.2940


In [ ]:
df.columns

Index(['issue_type', 'priority_numeric', 'status', 'component', 'story_points',
       'sprint', 'assignee_id', 'reporter_id', 'resolution', 'num_comments',
       'num_attachments', 'watchers_count', 'time_to_resolution_hours',
       'num_transitions', 'description_length', 'ticket_age_days',
       'has_labels', 'parent_epic_id', 'parent_story_id', 'has_child_stories',
       'has_child_subtasks', 'priority'],
      dtype='object')

## Model

In [ ]:
#X = X.drop(columns=['watchers_count', 'has_labels', 'num_attachments'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# XGBoost Model (Gradient Boosting)
xgboost_model = XGBRegressor(
    random_state=42,
    n_estimators=100,
    learning_rate=0.05,
    max_depth=10,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=4
)

In [ ]:
# Train XGBoost
print("\n\nTraining XGBoost Model...")
xgboost_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=30
)



Training XGBoost Model...
[0]	validation_0-rmse:19.18125
[9]	validation_0-rmse:19.09360


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=4,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
xgboost_pred = xgboost_model.predict(X_test)

# XGBoost Evaluation
xgboost_mse = mean_squared_error(y_test, xgboost_pred)
xgboost_rmse = np.sqrt(xgboost_mse)
xgboost_mae = mean_absolute_error(y_test, xgboost_pred)
xgboost_r2 = r2_score(y_test, xgboost_pred)

print("\n📊 XGBoost Regressor:")
print(f"  MSE:  {xgboost_mse:.4f}")
print(f"  RMSE: {xgboost_rmse:.4f}")
print(f"  MAE:  {xgboost_mae:.4f}")
print(f"  R²:   {xgboost_r2:.4f}")


📊 XGBoost Regressor:
  MSE:  362.5884
  RMSE: 19.0418
  MAE:  15.5742
  R²:   0.0155


In [ ]:
predictions = xgboost_model.predict(X)

## Output Predictions

In [ ]:
df_preds = pd.read_csv("jira_m2_accelerated.csv")

In [ ]:
df.columns

Index(['issue_type', 'priority_numeric', 'status', 'component', 'story_points',
       'sprint', 'assignee_id', 'reporter_id', 'resolution', 'num_comments',
       'num_attachments', 'watchers_count', 'time_to_resolution_hours',
       'num_transitions', 'description_length', 'ticket_age_days',
       'has_labels', 'parent_epic_id', 'parent_story_id', 'has_child_stories',
       'has_child_subtasks', 'priority'],
      dtype='object')

In [ ]:
df_preds['predictions'] = predictions
df.drop(columns=['status', 'component', 'assignee_id', 'reporter_id', 'resolution', 'num_comments',
       'num_attachments', 'watchers_count',
       'num_transitions', 'description_length', 'ticket_age_days',
       'has_labels', 'parent_epic_id', 'parent_story_id', 'has_child_stories',
       'has_child_subtasks'])

,issue_type,priority_numeric,story_points,sprint,time_to_resolution_hours,priority
0,1,5,3,17,41.09,5
1,4,5,5,14,71.18,5
2,3,3,3,6,45.39,3
3,5,4,1,13,25.08,4
4,3,5,5,8,88.71,5
...,...,...,...,...,...,...
1995,2,4,3,2,22.47,4
1996,2,3,2,4,104.21,3
1997,2,1,2,17,76.46,1
1998,2,2,2,20,94.66,2


In [ ]:
df_preds.to_csv('predictions.csv', index=False)